# Medical Assistant - Part 2: Fine-tuning

Fine-tune Phi-3 Mini on medical data using LoRA.

**Before starting:** Run `01_data_preparation.ipynb` first

## 1. Install Dependencies

In [1]:
!pip install --quiet datasets==4.8.5 pandas==3.0.2 transformers==5.8.0 peft==0.19.1 torch==2.11.0


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import torch
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling

print("✅ Setup complete")

/Users/anabarbosa/fiap/fiap-tech-challenge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Setup complete


## 2. Load Prepared Data

In [3]:
# Load training data
with open("data/processed/medical_data_train.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

# Load validation data
with open("data/processed/medical_data_val.jsonl", "r") as f:
    val_data = [json.loads(line) for line in f]

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nExample: {train_data[0]['text'][:200]}...")

Training samples: 168407
Validation samples: 42102

Example: Medication: 120 ACTUAT Fluticasone propionate 0.044 MG/ACTUAT Metered Dose Inhaler...


## 3. Load Model and Tokenizer

In [4]:
model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

print(f"✅ Model loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

Loading microsoft/Phi-3-mini-4k-instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:17<00:00, 11.16it/s]


✅ Model loaded
Parameters: 3.82B


## 4. Create HuggingFace Datasets

In [5]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict({
    "text": [item["text"] for item in train_data]
})

val_dataset = Dataset.from_dict({
    "text": [item["text"] for item in val_data]
})

print(f"Train dataset shape: {train_dataset.shape}")
print(f"Val dataset shape: {val_dataset.shape}")

# Tokenize the datasets
print("\nTokenizing datasets...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors=None
    )

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("✅ Tokenization complete")

Train dataset shape: (168407, 1)
Val dataset shape: (42102, 1)

Tokenizing datasets...


Map: 100%|██████████| 42102/42102 [00:08<00:00, 4773.59 examples/s]

✅ Tokenization complete


## 5. Configure LoRA

In [6]:
# LoRA Configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                                      # Rank (smaller = less memory)
    lora_alpha=16,                            # Scaling factor
    lora_dropout=0.05,                        # Regularization
    target_modules=["qkv_proj", "o_proj"],    # Phi-3 Mini attention layers
    bias="none"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Check trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
percent = 100 * trainable / total

print(f"✅ LoRA applied")
print(f"Trainable parameters: {trainable/1e6:.1f}M ({percent:.2f}%)")
print(f"Total parameters: {total/1e9:.2f}B")

✅ LoRA applied
Trainable parameters: 4.7M (0.12%)
Total parameters: 3.83B


## 6. Setup Training

In [7]:
# Training arguments
training_args = TrainingArguments(
    output_dir="outputs",
    num_train_epochs=2,              # Number of epochs
    per_device_train_batch_size=1,   # Batch size (reduced for M1 memory)
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,   # Accumulate gradients
    learning_rate=1e-4,              # Learning rate
    warmup_steps=100,                # Warmup
    logging_steps=50,                # Log every 50 steps
    eval_strategy="steps",           # Evaluate during training
    eval_steps=100,                  # Evaluate every 100 steps
    save_strategy="steps",           # Save checkpoints
    save_steps=200,
    seed=42,
    gradient_checkpointing=True,     # Enable gradient checkpointing for memory efficiency
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal language modeling
)

print("✅ Training configuration set")

✅ Training configuration set


## 7. Create Trainer

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("✅ Trainer created")

✅ Trainer created


## 8. Train!

In [ ]:
print("Starting training...")

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Final loss: {train_result.training_loss:.4f}")

Starting training...


/Users/anabarbosa/fiap/fiap-tech-challenge/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


## 9. Evaluate

In [ ]:
print("Evaluating on validation set...\n")

eval_results = trainer.evaluate()

print("Validation Results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

## 10. Save Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("models/medical_assistant_lora")
tokenizer.save_pretrained("models/medical_assistant_lora")

print("✅ Model saved to models/medical_assistant_lora")

## 11. Test the Model

In [ ]:
# Set model to inference mode
model.eval()

# Test prompts
test_prompts = [
    "What is hypertension?",
    "How do you treat diabetes?",
    "What are the symptoms of pneumonia?",
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    
    # Tokenize input
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    
    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    
    # Decode and print
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Response: {response}\n")
    print("-" * 70)

## Done!

✅ Fine-tuning complete!

What was trained:
- Model: Phi-3 Mini
- Data: ~210K medical samples
- Method: LoRA fine-tuning
- Epochs: 2
- Result: Medical assistant model

Model saved to: `models/medical_assistant_lora/`